In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
#Out-of-sample data; saved from 1_Preprocessing and 2_NN3
y_true = torch.load('y_true.pth').cpu()
y_pred = torch.load('y_pred_NN3.pth').cpu()
permno = np.load('permno.npy')
mdate = np.load('oos_periods.npy',allow_pickle=True)
mktval = np.load('mvel.npy')

In [3]:
#Aggregate stock-level monthly prediction
stock_level_prediction = pd.DataFrame({
    'mdate': mdate,
    'permno': permno,
    'y_pred': y_pred.ravel(),
    'y_true': y_true.ravel(),
    'mktval': mktval,
})
T = stock_level_prediction['mdate'].nunique()
stock_level_prediction['decile'] = stock_level_prediction.groupby('mdate')['y_pred'].transform(lambda x: pd.qcut(x,10,labels=False))
stock_level_prediction['sum_mval'] = stock_level_prediction.groupby(['mdate','decile'])['mktval'].transform('sum')

#Portfolio turnover
##Stock's weight in the long-short portfolios
###Multiplying with 0.5 to normalize the total weight (long+short) to sum to 1; assuming a zero-net investment where values of the long and short portfolios are equal
stock_level_prediction['ew_weight'] = 0.5 * 1/ stock_level_prediction['permno'].groupby([stock_level_prediction['mdate'],stock_level_prediction['decile']]).transform('count')
stock_level_prediction['vw_weight'] = 0.5 * stock_level_prediction['mktval']/stock_level_prediction['sum_mval']
stock_level_prediction.loc[stock_level_prediction['decile']==0,['ew_weight','vw_weight']] = -1 * stock_level_prediction.loc[stock_level_prediction['decile']==0,['ew_weight','vw_weight']]
stock_level_prediction.loc[~stock_level_prediction['decile'].isin([0,9]),['ew_weight','vw_weight']] = 0.0
stock_level_prediction['lagged_ew_weight'] = stock_level_prediction['ew_weight'].groupby(stock_level_prediction['permno']).shift(1)
stock_level_prediction['lagged_vw_weight'] = stock_level_prediction['vw_weight'].groupby(stock_level_prediction['permno']).shift(1)

ew_numer_lag = stock_level_prediction['lagged_ew_weight'] * (1 + stock_level_prediction['y_true'])
ew_portfret = (stock_level_prediction['lagged_ew_weight'] * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
ew_denom_lag = 1 + ew_portfret
ew_turnover = (stock_level_prediction['ew_weight'] - ew_numer_lag/ew_denom_lag).abs().sum() / T *100

vw_numer_lag = stock_level_prediction['lagged_vw_weight'] * (1 + stock_level_prediction['y_true'])
vw_portfret = (stock_level_prediction['lagged_vw_weight'] * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
vw_denom_lag = 1 + vw_portfret
vw_turnover = (stock_level_prediction['vw_weight'] - vw_numer_lag/vw_denom_lag).abs().sum() / T *100

#Equally weighted decile-sorted portfolios
ewportf_monthly = pd.DataFrame({
    'y_pred': stock_level_prediction.groupby(['mdate','decile'])['y_pred'].mean()*100,
    'y_true': stock_level_prediction.groupby(['mdate','decile'])['y_true'].mean()*100,
})
ewportf_monthly = ewportf_monthly.reset_index(drop=False)
ewportf_monthly_avg = pd.DataFrame({
    'y_pred_avg': ewportf_monthly.groupby(['decile'])['y_pred'].mean(),
    'y_true_avg': ewportf_monthly.groupby(['decile'])['y_true'].mean(),
    'y_true_std': ewportf_monthly.groupby(['decile'])['y_true'].std(),
})
ewportf_monthly_avg['sharpe_ratio (annualized)'] = (ewportf_monthly_avg['y_true_avg'] * 12) / np.sqrt(ewportf_monthly_avg['y_true_std']**2 * 12)
ewportf_high_minus_low = ewportf_monthly.pivot(index='mdate',columns='decile',values='y_true').reset_index(drop=False).rename_axis(None, axis=1)
ewportf_high_minus_low['H-L'] = ewportf_high_minus_low[9] - ewportf_high_minus_low[0]

#Value weighted decile-sorted portfolios
vwportf_monthly = pd.DataFrame({
    'y_pred': (stock_level_prediction['y_pred'] * stock_level_prediction['mktval'] / stock_level_prediction['sum_mval']).groupby([stock_level_prediction['mdate'],stock_level_prediction['decile']]).sum()*100,
    'y_true': (stock_level_prediction['y_true'] * stock_level_prediction['mktval'] / stock_level_prediction['sum_mval']).groupby([stock_level_prediction['mdate'],stock_level_prediction['decile']]).sum()*100
})
vwportf_monthly = vwportf_monthly.reset_index(drop=False)
vwportf_monthly_avg = pd.DataFrame({
    'y_pred_avg': vwportf_monthly.groupby(['decile'])['y_pred'].mean(),
    'y_true_avg': vwportf_monthly.groupby(['decile'])['y_true'].mean(),
    'y_true_std': vwportf_monthly.groupby(['decile'])['y_true'].std(),
})
vwportf_monthly_avg['sharpe_ratio (annualized)'] = (vwportf_monthly_avg['y_true_avg'] * 12) / np.sqrt(vwportf_monthly_avg['y_true_std']**2 * 12)
vwportf_high_minus_low = vwportf_monthly.pivot(index='mdate',columns='decile',values='y_true').reset_index(drop=False).rename_axis(None, axis=1)
vwportf_high_minus_low['H-L'] = vwportf_high_minus_low[9] - vwportf_high_minus_low[0]

#Long-short porfolios: Long highest decile, short lowest decile
longshort_portfolio = pd.DataFrame(
    np.array([
        [ewportf_monthly_avg['y_pred_avg'][9] - ewportf_monthly_avg['y_pred_avg'][0],
        ewportf_monthly_avg['y_true_avg'][9] - ewportf_monthly_avg['y_true_avg'][0],
        ewportf_high_minus_low['H-L'].std(axis=0)],
        [vwportf_monthly_avg['y_pred_avg'][9] - vwportf_monthly_avg['y_pred_avg'][0],
        vwportf_monthly_avg['y_true_avg'][9] - vwportf_monthly_avg['y_true_avg'][0],
        vwportf_high_minus_low['H-L'].std(axis=0)],
               ]),
    columns=['y_pred_avg', 'y_true_avg','y_true_std'],
    index=['equally weighted', 'value weighted']
).reset_index(drop=False)
longshort_portfolio['sharpe_ratio (annualized)'] = (longshort_portfolio['y_true_avg'] * 12) / np.sqrt((longshort_portfolio['y_true_std'])**2 * 12)

#Maximum 1M loss and maximum drawdown
ew_monthly_logret = np.log(ewportf_high_minus_low['H-L']/100+1)*100
ew_cummulative_logret = np.cumsum(ew_monthly_logret)
vw_monthly_logret = np.log(vwportf_high_minus_low['H-L']/100+1)*100
vw_cummulative_logret = np.cumsum(vw_monthly_logret)
ew_drawdown = - float('inf')
vw_drawdown = - float('inf')
for i in range(T):
    for j in range(i+1,T):
        ew_dd = ew_cummulative_logret[i] - ew_cummulative_logret[j]
        vw_dd = vw_cummulative_logret[i] - vw_cummulative_logret[j]
        if ew_dd > ew_drawdown:
            ew_drawdown = ew_dd
        if vw_dd > vw_drawdown:
            vw_drawdown = vw_dd

In [4]:
Table_7_Performance_of_value_weighted_MLportfolios_NN3 = pd.DataFrame(
    data=np.vstack((vwportf_monthly_avg.values, longshort_portfolio.iloc[1,1:].values.reshape(1,-1))),
    columns=longshort_portfolio.iloc[:,1:].columns,
    index = [f'decile {i}' for i in range(1,11)] + ['H-L']
)
Table_7_Performance_of_value_weighted_MLportfolios_NN3

,y_pred_avg,y_true_avg,y_true_std,sharpe_ratio (annualized)
decile 1,-1.038993,-0.111366,7.20731,-0.053527
decile 2,-0.159789,0.177794,5.724389,0.107591
decile 3,0.212519,0.530893,4.900385,0.37529
decile 4,0.453711,0.670147,4.417829,0.525475
decile 5,0.641187,0.580364,4.294373,0.468157
decile 6,0.804905,0.757014,4.200695,0.624271
decile 7,0.95612,0.717624,4.236087,0.586844
decile 8,1.108322,0.822181,4.289757,0.663935
decile 9,1.285981,0.992334,4.572424,0.751799
decile 10,1.955849,1.542205,7.263493,0.735508


In [9]:
Table_7_Comparison_with_Originalpaper = pd.DataFrame(
    Table_7_Performance_of_value_weighted_MLportfolios_NN3.iloc[[0,-2,-1],:].stack(),
    columns=['Replicated']
)
Table_7_Comparison_with_Originalpaper['Original'] = pd.Series([-0.03, -0.43, 7.73, -0.19, 1.83, 1.69, 7.29, 0.80, 1.86, 2.12, 6.13, 1.20],
                                                              index=Table_7_Comparison_with_Originalpaper.index)
Table_7_Comparison_with_Originalpaper['Rep. quality'] = pd.Series(['not good','not good','good','good','good','good','good','good','not good','not good','good','good'],
                                                                  index=Table_7_Comparison_with_Originalpaper.index)
Table_7_Comparison_with_Originalpaper

Replicated  Original Rep. quality
decile 1  y_pred_avg                 -1.038993     -0.03     not good
          y_true_avg                 -0.111366     -0.43     not good
          y_true_std                   7.20731      7.73         good
          sharpe_ratio (annualized)  -0.053527     -0.19         good
decile 10 y_pred_avg                  1.955849      1.83         good
          y_true_avg                  1.542205      1.69         good
          y_true_std                  7.263493      7.29         good
          sharpe_ratio (annualized)   0.735508      0.80         good
H-L       y_pred_avg                  2.994841      1.86     not good
          y_true_avg                  1.653571      2.12     not good
          y_true_std                   5.10378      6.13         good
          sharpe_ratio (annualized)   1.122332      1.20         good

In [6]:
Table_A9_Performance_of_equally_weighted_MLportfolios_NN3 = pd.DataFrame(
    data=np.vstack((ewportf_monthly_avg.values, longshort_portfolio.iloc[0,1:].values.reshape(1,-1))),
    columns=longshort_portfolio.iloc[:,1:].columns,
    index = [f'decile {i}' for i in range(1,11)] + ['H-L']
)
Table_A9_Performance_of_equally_weighted_MLportfolios_NN3

,y_pred_avg,y_true_avg,y_true_std,sharpe_ratio (annualized)
decile 1,-1.194421,-0.737084,7.953186,-0.321045
decile 2,-0.174624,0.220795,6.607714,0.115752
decile 3,0.208873,0.401116,5.528425,0.251339
decile 4,0.452715,0.661029,4.974296,0.460341
decile 5,0.641454,0.762634,4.503151,0.586665
decile 6,0.804415,0.814854,4.3355,0.651075
decile 7,0.956998,0.934226,4.436823,0.729408
decile 8,1.111437,1.032062,4.748962,0.752831
decile 9,1.311148,1.305288,5.237342,0.863348
decile 10,2.708584,2.492451,8.44355,1.022568


In [7]:
Table_A9_Comparison_with_Originalpaper = pd.DataFrame(
    Table_A9_Performance_of_equally_weighted_MLportfolios_NN3.iloc[[0,-2,-1],:].stack(),
    columns=['Replicated']
)
Table_A9_Comparison_with_Originalpaper['Original'] = pd.Series([-0.31, -0.92, 7.94, -0.40, 2.28, 2.35, 8.11, 1.00, 2.58, 3.27, 4.80, 2.36],
                                                index=Table_A9_Comparison_with_Originalpaper.index)
Table_A9_Comparison_with_Originalpaper['Rep. quality'] = pd.Series(['not good', 'not good', 'good', 'good', 'good', 'good', 'good', 'good', 'good' ,'good', 'good', 'good'],
                                                index=Table_A9_Comparison_with_Originalpaper.index)
Table_A9_Comparison_with_Originalpaper

Replicated  Original Rep. quality
decile 1  y_pred_avg                 -1.194421     -0.31     not good
          y_true_avg                 -0.737084     -0.92     not good
          y_true_std                  7.953186      7.94         good
          sharpe_ratio (annualized)  -0.321045     -0.40         good
decile 10 y_pred_avg                  2.708584      2.28         good
          y_true_avg                  2.492451      2.35         good
          y_true_std                   8.44355      8.11         good
          sharpe_ratio (annualized)   1.022568      1.00         good
H-L       y_pred_avg                  3.903005      2.58         good
          y_true_avg                  3.229534      3.27         good
          y_true_std                  4.476533      4.80         good
          sharpe_ratio (annualized)   2.499129      2.36         good

In [8]:
Table_8_Drawdown_Turnover_of_MLportfolios_NN3 = pd.DataFrame(
    data=np.vstack(([vw_drawdown, 30.84], [vw_monthly_logret.min()*-1, 30.84], [vw_turnover, 123.50],
           [ew_drawdown, 17.34], [ew_monthly_logret.min()*-1, 12.50], [ew_turnover, 113.76])),
    columns=['Replicated', 'Original'],
    index=pd.MultiIndex.from_product([['Value weighted', 'Equally weighted'],('Max drawdown (%)', 'Max 1M loss (%)', 'Turnover (%)')])
)
Table_8_Drawdown_Turnover_of_MLportfolios_NN3['Rep. quality'] = pd.Series(['good','good','good','good','good','good'],
                                                                          index=Table_8_Drawdown_Turnover_of_MLportfolios_NN3.index)
Table_8_Drawdown_Turnover_of_MLportfolios_NN3

Replicated  Original Rep. quality
Value weighted   Max drawdown (%)   26.784452     30.84         good
                 Max 1M loss (%)    13.490679     30.84         good
                 Turnover (%)      128.458104    123.50         good
Equally weighted Max drawdown (%)    8.880981     17.34         good
                 Max 1M loss (%)     8.094756     12.50         good
                 Turnover (%)      116.664008    113.76         good